# BAN 612 — Week 3 In-Class Practice: Web Scraping

**Topic:** Turn web pages into structured data.

This notebook uses a small **local HTML example** so that we can run even if the internet or a website is unavailable. It also includes an optional live scraping section using Books to Scrape, a public practice website.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the difference between downloading a page and parsing HTML.
2. Use `requests` to obtain web content.
3. Use `BeautifulSoup` to parse HTML.
4. Use CSS selectors to locate repeated records.
5. Extract text and attributes from HTML elements.
6. Clean scraped strings into usable variables.
7. Handle missing fields defensively.
8. Loop across multiple pages.
9. Validate and save the final scraped dataset.


## 0. Setup

We use four main packages:

- `pandas`: stores the final scraped data as a DataFrame.
- `BeautifulSoup` from `bs4`: parses HTML and lets us search/select elements.
- `re`: supports regular expressions for extracting patterns from text.
- `time`: allows us to pause between web requests when scraping live pages.
- `urljoin`: safely combines a base URL and a relative link.

`requests` is used in the optional live scraping section. It sends HTTP requests to a website and receives the response.


In [1]:
# Core data analysis package
import pandas as pd

# HTML parser. BeautifulSoup turns raw HTML text into a searchable object.
from bs4 import BeautifulSoup

# Regular expression package for extracting patterns from strings.
import re

# Used to pause between web requests. Helpful for polite scraping.
import time

# Used to combine base URLs with relative links such as "catalogue/page-2.html".
from urllib.parse import urljoin

# Optional: used for live web requests.
import requests

# Show more rows/columns during demonstration.
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)


## 1. A tiny HTML page

A web page is just text written in HTML. The page below contains three repeated product cards.

Each product card contains product title, URL, price, rating, and availability. But the data are not in a table yet. They are embedded in HTML tags and attributes.


In [2]:
html_page_1 = """
<html>
  <head><title>Training Store - Page 1</title></head>
  <body>
    <h1>Training Store</h1>

    <article class="product-card">
      <h3><a class="product-link" href="/products/notebook.html" title="Analyst Notebook">Analyst Notebook</a></h3>
      <p class="price">$12.99</p>
      <p class="rating rating-4">4 stars</p>
      <p class="availability">In stock (22 available)</p>
    </article>

    <article class="product-card">
      <h3><a class="product-link" href="/products/keyboard.html" title="Wireless Keyboard">Wireless Keyboard</a></h3>
      <p class="price">$49.95</p>
      <p class="rating rating-5">5 stars</p>
      <p class="availability">Only 3 left in stock</p>
    </article>

    <article class="product-card">
      <h3><a class="product-link" href="/products/mug.html" title="Travel Mug">Travel Mug</a></h3>
      <p class="rating rating-3">3 stars</p>
      <p class="availability">Out of stock</p>
    </article>

    <a class="next" href="page2.html">Next</a>
  </body>
</html>
"""

# Display the first 500 characters so we can see that this is just text.
print(html_page_1[:500])



<html>
  <head><title>Training Store - Page 1</title></head>
  <body>
    <h1>Training Store</h1>

    <article class="product-card">
      <h3><a class="product-link" href="/products/notebook.html" title="Analyst Notebook">Analyst Notebook</a></h3>
      <p class="price">$12.99</p>
      <p class="rating rating-4">4 stars</p>
      <p class="availability">In stock (22 available)</p>
    </article>

    <article class="product-card">
      <h3><a class="product-link" href="/products/keyboard.ht


## 2. Parse HTML with BeautifulSoup

`BeautifulSoup(html_text, "html.parser")` creates a parsed representation of the HTML.

Syntax:

```python
soup = BeautifulSoup(html_text, "html.parser")
```

Important parameters:

- `html_text`: the string containing HTML.
- `"html.parser"`: Python's built-in HTML parser. It is convenient because it does not require installing another parser.

After parsing, we can search the document using tags, classes, attributes, and CSS selectors.


In [3]:
# Parse the HTML string into a BeautifulSoup object.
soup = BeautifulSoup(html_page_1, "html.parser")

# .prettify() prints the HTML in an indented way.
# This is useful for inspection, but not usually used in final production code.
print(soup.prettify()[:1000])
#html.parser is fine for class; checkout lxml which may be faster but requires another dependency.


<html>
 <head>
  <title>
   Training Store - Page 1
  </title>
 </head>
 <body>
  <h1>
   Training Store
  </h1>
  <article class="product-card">
   <h3>
    <a class="product-link" href="/products/notebook.html" title="Analyst Notebook">
     Analyst Notebook
    </a>
   </h3>
   <p class="price">
    $12.99
   </p>
   <p class="rating rating-4">
    4 stars
   </p>
   <p class="availability">
    In stock (22 available)
   </p>
  </article>
  <article class="product-card">
   <h3>
    <a class="product-link" href="/products/keyboard.html" title="Wireless Keyboard">
     Wireless Keyboard
    </a>
   </h3>
   <p class="price">
    $49.95
   </p>
   <p class="rating rating-5">
    5 stars
   </p>
   <p class="availability">
    Only 3 left in stock
   </p>
  </article>
  <article class="product-card">
   <h3>
    <a class="product-link" href="/products/mug.html" title="Travel Mug">
     Travel Mug
    </a>
   </h3>
   <p class="rating rating-3">
    3 stars
   </p>
   <p class="availab

## 3. Find elements

BeautifulSoup provides several ways to locate HTML elements.

- `.find()` returns the first matching element.
- `.find_all()` returns all matching elements.
- `.select()` uses CSS (Cascading Style Sheets) selector syntax. This is often the most convenient method for scraping repeated page structures.

Common CSS selector patterns:

| Selector | Meaning |
|---|---|
| `article` | all `<article>` tags |
| `.product-card` | any element with class `product-card` |
| `article.product-card` | `<article>` tags with class `product-card` |
| `article.product-card .price` | elements with class `price` inside product cards |
| `h3 a` | `<a>` tags inside `<h3>` tags |


In [4]:
# First h1 element
print(soup.find("h1"))

<h1>Training Store</h1>


In [5]:
# All product cards using CSS selector syntax.
product_cards = soup.select("article.product-card")
print("Number of product cards:", len(product_cards))

# Show the first product card.
print(product_cards[0].prettify())

Number of product cards: 3
<article class="product-card">
 <h3>
  <a class="product-link" href="/products/notebook.html" title="Analyst Notebook">
   Analyst Notebook
  </a>
 </h3>
 <p class="price">
  $12.99
 </p>
 <p class="rating rating-4">
  4 stars
 </p>
 <p class="availability">
  In stock (22 available)
 </p>
</article>



## 4. Extract text from an element

`.get_text()` extracts the visible text inside an HTML element.

Syntax:

```python
element.get_text(strip=True)
```

Important parameter:

- `strip=True`: removes leading and trailing whitespace.

Without `strip=True`, scraped text often contains newlines or extra spaces.


In [9]:
first_card = product_cards[0]

# Select the price element inside the first product card.
price_element = first_card.select_one(".price")
price_element
print("HTML element:", price_element)



HTML element: <p class="price">$12.99</p>


In [10]:
# Extract text from the element.
raw_price = price_element.get_text(strip=True)


print("Extracted text:", raw_price)

Extracted text: $12.99


## 5. Extract attributes

HTML elements can contain attributes such as `href`, `title`, `class`, and `id`.

Syntax:

```python
element["href"]
element.get("href")
```

Difference:

- `element["href"]` raises an error if the attribute is missing.
- `element.get("href")` returns `None` if the attribute is missing.


In [12]:
link_element = first_card.select_one("h3 a")
link_element


<a class="product-link" href="/products/notebook.html" title="Analyst Notebook">Analyst Notebook</a>

In [13]:
visible_title = link_element.get_text(strip=True)
title_attribute = link_element.get("title")
relative_url = link_element.get("href")

print("Visible title:", visible_title)
print("Title attribute:", title_attribute)
print("Relative URL:", relative_url)

Visible title: Analyst Notebook
Title attribute: Analyst Notebook
Relative URL: /products/notebook.html


## 6. Convert a relative URL to a full URL

Many HTML links are relative. For example `/products/notebook.html` is not a complete URL. We need a base URL.

`urljoin(base_url, relative_url)` combines them safely.


In [14]:
base_url = "https://example-training-store.com"
full_url = urljoin(base_url, relative_url)

print(full_url)


https://example-training-store.com/products/notebook.html


## 7. Clean a price string

Scraped prices usually arrive as strings. For analysis, we need a numeric value.

`pd.to_numeric(..., errors="coerce")` is often safer than `float(...)` for a Series.

Important parameter:

- `errors="coerce"`: if a value cannot be converted, return `NaN` instead of stopping the program.


In [11]:
raw_prices = pd.Series(["$12.99", "$1,249.50", "Price unavailable"])

clean_prices = (
    raw_prices
    .str.replace("$", "", regex=False)   # literal replacement: remove dollar sign
    .str.replace(",", "", regex=False)   # literal replacement: remove comma
)

numeric_prices = pd.to_numeric(clean_prices, errors="coerce")

pd.DataFrame({"raw": raw_prices, "clean_string": clean_prices, "numeric": numeric_prices})


,raw,clean_string,numeric
0,$12.99,12.99,12.99
1,"$1,249.50",1249.50,1249.50
2,Price unavailable,Price unavailable,NaN


## 8. Extract rating from text using regex

`r"\d+"` means **one or more digits**.

- `\d` = a digit
- `+` = one or more
- `r"..."` = raw string, which is convenient for regex in Python

`re.search(pattern, text)` looks for the first match.


In [9]:
rating_element = first_card.select_one(".rating")
rating_text = rating_element.get_text(strip=True)

match = re.search(r"\d+", rating_text)

if match:
    rating = int(match.group())
else:
    rating = None

print("Rating text:", rating_text)
print("Rating numeric:", rating)


Rating text: 4 stars
Rating numeric: 4


## 9. Defensive extraction

Naive code often assumes every element exists. Real web pages are inconsistent.

For example, the third product card has no price. If we run `card.select_one(".price").get_text(strip=True)` on that card, it will fail because `select_one(".price")` returns `None`.

A safer approach is to write helper functions.


In [15]:
def text_or_none(element, selector):
    # Return stripped text for the first matching CSS selector.
    # element: BeautifulSoup element to search inside.
    # selector: CSS selector, such as '.price' or 'h3 a'.
    # Returns: clean text if the element exists; None otherwise.
    found = element.select_one(selector)
    if found is None:
        return None
    return found.get_text(strip=True)


def attr_or_none(element, selector, attr):
    # Return an attribute value from the first matching CSS selector.
    # attr can be 'href', 'title', 'class', etc.
    found = element.select_one(selector)
    if found is None:
        return None
    return found.get(attr)


## 10. Write functions to parse one product card

A good scraper separates repeated logic into functions. This function takes one product card and returns one row dictionary.


In [16]:
def parse_price(raw_price):
    # Convert a scraped price string into a numeric value.
    # Examples: '$12.99' -> 12.99; '$1,249.50' -> 1249.50; None -> None.
    if raw_price is None:
        return None
    cleaned = raw_price.replace("$", "").replace(",", "")
    try:
        return float(cleaned)
    except ValueError:
        return None


def parse_rating(raw_rating):
    # Extract a numeric rating from text such as '4 stars'.
    if raw_rating is None:
        return None
    match = re.search(r"\d+", raw_rating)
    if match is None:
        return None
    return int(match.group())


def parse_available_count(raw_availability):
    # Extract available inventory count from availability text.
    # Examples: 'In stock (22 available)' -> 22; 'Only 3 left in stock' -> 3; 'Out of stock' -> 0.
    if raw_availability is None:
        return None
    text = raw_availability.lower()
    if "out of stock" in text:
        return 0
    match = re.search(r"\d+", raw_availability)
    if match:
        return int(match.group())
    return None


def parse_product_card(card, base_url):
    # Extract one product card into a row dictionary.
    raw_title = attr_or_none(card, "h3 a", "title")
    raw_url = attr_or_none(card, "h3 a", "href")
    raw_price = text_or_none(card, ".price")
    raw_rating = text_or_none(card, ".rating")
    raw_availability = text_or_none(card, ".availability")

    return {
        "title": raw_title,
        "url": urljoin(base_url, raw_url) if raw_url else None,
        "raw_price": raw_price,
        "price": parse_price(raw_price),
        "raw_rating": raw_rating,
        "rating": parse_rating(raw_rating),
        "raw_availability": raw_availability,
        "available_count": parse_available_count(raw_availability),
    }


## 11. Parse all cards on one page

The pattern is:

1. Select all repeated product cards.
2. Loop over each card.
3. Parse each card into a dictionary.
4. Store dictionaries in a list.
5. Convert the list into a DataFrame.


In [17]:
rows = []

for card in soup.select("article.product-card"):
    row = parse_product_card(card, base_url=base_url)
    rows.append(row)

products_page_1 = pd.DataFrame(rows)
products_page_1


,title,url,raw_price,price,raw_rating,rating,raw_availability,available_count
0,Analyst Notebook,https://example-training-store.com/products/no...,$12.99,12.99,4 stars,4,In stock (22 available),22
1,Wireless Keyboard,https://example-training-store.com/products/ke...,$49.95,49.95,5 stars,5,Only 3 left in stock,3
2,Travel Mug,https://example-training-store.com/products/mu...,None,NaN,3 stars,3,Out of stock,0


In [18]:
rows

[{'title': 'Analyst Notebook',
  'url': 'https://example-training-store.com/products/notebook.html',
  'raw_price': '$12.99',
  'price': 12.99,
  'raw_rating': '4 stars',
  'rating': 4,
  'raw_availability': 'In stock (22 available)',
  'available_count': 22},
 {'title': 'Wireless Keyboard',
  'url': 'https://example-training-store.com/products/keyboard.html',
  'raw_price': '$49.95',
  'price': 49.95,
  'raw_rating': '5 stars',
  'rating': 5,
  'raw_availability': 'Only 3 left in stock',
  'available_count': 3},
 {'title': 'Travel Mug',
  'url': 'https://example-training-store.com/products/mug.html',
  'raw_price': None,
  'price': None,
  'raw_rating': '3 stars',
  'rating': 3,
  'raw_availability': 'Out of stock',
  'available_count': 0}]

## 12. Validate the one-page result

Before collecting more pages, validate the result on one page.


In [19]:
print("Rows collected:", len(products_page_1))
print("\nMissing values:")
print(products_page_1.isna().sum())

print("\nData types:")
print(products_page_1.dtypes)

print("\nRating values:")
print(products_page_1["rating"].value_counts(dropna=False).sort_index())


Rows collected: 3

Missing values:
title               0
url                 0
raw_price           1
price               1
raw_rating          0
rating              0
raw_availability    0
available_count     0
dtype: int64

Data types:
title                object
url                  object
raw_price            object
price               float64
raw_rating           object
rating                int64
raw_availability     object
available_count       int64
dtype: object

Rating values:
rating
3    1
4    1
5    1
Name: count, dtype: int64


## 13. Add a second page and pagination

Real scraping often requires multiple pages. Here we create a second local HTML page. In a live scrape, the second page would be downloaded from the web.


In [20]:
html_page_2 = """
<html>
  <head><title>Training Store - Page 2</title></head>
  <body>
    <h1>Training Store</h1>

    <article class="product-card">
      <h3><a class="product-link" href="/products/calculator.html" title="Financial Calculator">Financial Calculator</a></h3>
      <p class="price">$89.00</p>
      <p class="rating rating-5">5 stars</p>
      <p class="availability">In stock (7 available)</p>
    </article>

    <article class="product-card">
      <h3><a class="product-link" href="/products/laptop-stand.html" title="Laptop Stand">Laptop Stand</a></h3>
      <p class="price">$34.50</p>
      <p class="rating rating-4">4 stars</p>
      <p class="availability">In stock (15 available)</p>
    </article>

    <article class="product-card">
      <h3><a class="product-link" href="/products/keyboard.html" title="Wireless Keyboard">Wireless Keyboard</a></h3>
      <p class="price">$49.95</p>
      <p class="rating rating-5">5 stars</p>
      <p class="availability">Only 2 left in stock</p>
    </article>
  </body>
</html>
"""

# Simulate a tiny website with two pages.
local_site = {
    "page1.html": html_page_1,
    "page2.html": html_page_2,
}


## 14. Follow the next link

The first page has `<a class="next" href="page2.html">Next</a>`. We can find this link and use it to decide whether another page exists.


In [21]:
next_link = soup.select_one("a.next")

print("Next link element:", next_link)
print("Next href:", next_link.get("href") if next_link else None)


Next link element: <a class="next" href="page2.html">Next</a>
Next href: page2.html


## 15. Parse multiple local pages

This function simulates pagination using the local HTML pages. In a real scrape, this same logic would use `requests.get()` to download each page.


In [22]:
def parse_local_site(start_page="page1.html", base_url="https://example-training-store.com"):
    # Parse multiple local HTML pages by following the 'next' link.
    rows = []
    current_page = start_page

    while current_page is not None:
        html = local_site[current_page]
        soup = BeautifulSoup(html, "html.parser")

        for card in soup.select("article.product-card"):
            row = parse_product_card(card, base_url=base_url)
            row["source_page"] = current_page
            rows.append(row)

        next_link = soup.select_one("a.next")
        current_page = next_link.get("href") if next_link else None

    return pd.DataFrame(rows)

products = parse_local_site()
products


,title,url,raw_price,price,raw_rating,rating,raw_availability,available_count,source_page
0,Analyst Notebook,https://example-training-store.com/products/no...,$12.99,12.99,4 stars,4,In stock (22 available),22,page1.html
1,Wireless Keyboard,https://example-training-store.com/products/ke...,$49.95,49.95,5 stars,5,Only 3 left in stock,3,page1.html
2,Travel Mug,https://example-training-store.com/products/mu...,None,NaN,3 stars,3,Out of stock,0,page1.html
3,Financial Calculator,https://example-training-store.com/products/ca...,$89.00,89.00,5 stars,5,In stock (7 available),7,page2.html
4,Laptop Stand,https://example-training-store.com/products/la...,$34.50,34.50,4 stars,4,In stock (15 available),15,page2.html
5,Wireless Keyboard,https://example-training-store.com/products/ke...,$49.95,49.95,5 stars,5,Only 2 left in stock,2,page2.html


## 16. Detect duplicates

The `Wireless Keyboard` appears on both pages. This could be a legitimate repeated listing or a scraping/pagination problem. We should not blindly drop duplicates without deciding what counts as the same item.


In [17]:
# Check duplicates by URL.
products["duplicate_url"] = products.duplicated(subset=["url"], keep=False)

products.loc[products["duplicate_url"], ["title", "url", "price", "available_count", "source_page"]]


,title,url,price,available_count,source_page
1,Wireless Keyboard,https://example-training-store.com/products/ke...,49.95,3,page1.html
5,Wireless Keyboard,https://example-training-store.com/products/ke...,49.95,2,page2.html


## 17. Data-quality issue log

Just as in Week 2, we can summarize issues in a small table. This helps document what happened during scraping and cleaning.


In [18]:
issue_log = pd.DataFrame([
    {
        "issue": "Missing price",
        "count": int(products["price"].isna().sum()),
        "recommended_action": "Flag and inspect product page; do not invent price"
    },
    {
        "issue": "Duplicate product URL",
        "count": int(products["duplicate_url"].sum()),
        "recommended_action": "Check whether duplicate is real or pagination artifact"
    },
    {
        "issue": "Invalid rating",
        "count": int(((~products["rating"].between(1, 5)) & products["rating"].notna()).sum()),
        "recommended_action": "Inspect rating extraction rule"
    },
])

issue_log


,issue,count,recommended_action
0,Missing price,1,Flag and inspect product page; do not invent p...
1,Duplicate product URL,2,Check whether duplicate is real or pagination ...
2,Invalid rating,0,Inspect rating extraction rule


## 18. Save the scraped dataset

Once we have a structured DataFrame, we can save it like any other dataset. `index=False` prevents Pandas from writing the row index as an extra column.


In [19]:
products.to_csv("week3_scraped_products_local.csv", index=False)
issue_log.to_csv("week3_scraping_issue_log.csv", index=False)

print("Saved week3_scraped_products_local.csv")
print("Saved week3_scraping_issue_log.csv")


Saved week3_scraped_products_local.csv
Saved week3_scraping_issue_log.csv


# Optional live demo: Books to Scrape

The following section uses [Books to Scrape](https://books.toscrape.com/), a website designed for scraping practice.

Run this section only if you have internet access.


## 19. Download one live page with `requests`

Syntax:

```python
response = requests.get(url, headers=headers, timeout=10)
```

Parameters:

- `url`: page to request.
- `headers`: optional dictionary with request metadata such as `User-Agent`.
- `timeout`: maximum number of seconds to wait before stopping.

Important attributes:

- `response.status_code`: HTTP status code.
- `response.text`: response content decoded as text.
- `response.content`: response content as raw bytes.


In [23]:
live_url = "https://books.toscrape.com/"

headers = {
    "User-Agent": "BAN612 teaching demo; small number of requests"
}

try:
    response = requests.get(live_url, headers=headers, timeout=10)
    print("Status code:", response.status_code)
    print("First 300 characters:")
    print(response.text[:300])
except requests.RequestException as e:
    print("Request failed:", e)


Status code: 200
First 300 characters:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lan


## 20. Parse the live page if available

We check the status code before parsing. If the request failed or returned a non-200 status, do not parse as if it were a successful page.


In [24]:
if 'response' in globals() and response.status_code == 200:
    live_soup = BeautifulSoup(response.text, "html.parser")
    live_cards = live_soup.select("article.product_pod")
    print("Number of book cards:", len(live_cards))
    print(live_cards[0].prettify()[:1000])
else:
    print("Live page not available. Continue using local HTML demo.")


Number of book cards: 20
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



## 21. Parse a Books to Scrape card

Books to Scrape uses a slightly different structure:

- product cards: `article.product_pod`
- title/link: `h3 a`
- price: `.price_color`
- availability: `.availability`
- rating class: `.star-rating`

The rating value is encoded in a class such as `One`, `Two`, `Three`, `Four`, or `Five`.


In [56]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

def parse_book_card(card, page_url):
    # Parse one Books to Scrape product card.
    link = card.select_one("h3 a")
    price_text = text_or_none(card, ".price_color")
    availability_text = text_or_none(card, ".availability")
    rating_element = card.select_one(".star-rating")

    title = link.get("title") if link else None
    href = link.get("href") if link else None
    url = urljoin(page_url, href) if href else None

    if price_text is not None:
        price_clean = price_text.replace("Â£", "")
        price = pd.to_numeric(price_clean, errors="coerce")
    else:
        price = None

    rating = None
    if rating_element is not None:
        classes = rating_element.get("class", [])
        for c in classes:
            if c in rating_map:
                rating = rating_map[c]

    return {
        "title": title,
        "url": url,
        "price_gbp": price,
        "raw price": price_text,
        "rating": rating,
        "availability": " ".join(availability_text.split()) if availability_text else None,       
    }


## 22. Scrape one live page into a DataFrame

In [57]:
if 'live_cards' in globals() and len(live_cards) > 0:
    live_rows = [parse_book_card(card, live_url) for card in live_cards]
    books_one_page = pd.DataFrame(live_rows)
    display(books_one_page.head())
    print(books_one_page.info())
else:
    print("Live book cards not available.")


,title,url,price_gbp,raw price,rating,availability
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,51.77,Â£51.77,3,In stock
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...,53.74,Â£53.74,1,In stock
2,Soumission,https://books.toscrape.com/catalogue/soumissio...,50.10,Â£50.10,1,In stock
3,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,47.82,Â£47.82,4,In stock
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,54.23,Â£54.23,5,In stock


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         20 non-null     object 
 1   url           20 non-null     object 
 2   price_gbp     20 non-null     float64
 3   raw price     20 non-null     object 
 4   rating        20 non-null     int64  
 5   availability  20 non-null     object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.1+ KB
None


## 23. Optional: scrape multiple live pages politely

This demonstration limits itself to a small number of pages.

Parameters:

- `max_pages`: number of pages to request.
- `sleep_seconds`: pause between requests.


In [58]:
def scrape_books_to_scrape(max_pages=2, sleep_seconds=1.0):
    # Scrape a small number of pages from Books to Scrape.
    rows = []
    current_url = "https://books.toscrape.com/catalogue/page-1.html"
    headers = {"User-Agent": "BAN612 teaching demo; small number of requests"}

    for page_num in range(max_pages):
        print("Requesting:", current_url)
        try:
            r = requests.get(current_url, headers=headers, timeout=10)
        except requests.RequestException as e:
            print("Request failed:", e)
            break

        if r.status_code != 200:
            print("Non-success status code:", r.status_code)
            break

        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article.product_pod")
        print("Cards found:", len(cards))

        for card in cards:
            row = parse_book_card(card, current_url)
            row["source_page"] = page_num + 1
            rows.append(row)

        next_link = soup.select_one("li.next a")
        if next_link is None:
            break

        current_url = urljoin(current_url, next_link.get("href"))
        time.sleep(sleep_seconds)

    return pd.DataFrame(rows)

# Uncomment the next line to run the live multi-page scrape.
# books = scrape_books_to_scrape(max_pages=2, sleep_seconds=1.0)
# books.head()


In [61]:
books=scrape_books_to_scrape(max_pages=2, sleep_seconds=1.0)
books.head()

Requesting: https://books.toscrape.com/catalogue/page-1.html
Cards found: 20
Requesting: https://books.toscrape.com/catalogue/page-2.html
Cards found: 20


,title,url,price_gbp,raw price,rating,availability,source_page
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,51.77,Â£51.77,3,In stock,1
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...,53.74,Â£53.74,1,In stock,1
2,Soumission,https://books.toscrape.com/catalogue/soumissio...,50.10,Â£50.10,1,In stock,1
3,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,47.82,Â£47.82,4,In stock,1
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,54.23,Â£54.23,5,In stock,1


# In-class exercises

Use the live `books` DataFrame.


## Exercise 1

Show all products with missing price. What should the analyst do with these records?

In [44]:
# Your code here
missing_price = books.loc[books["price_gbp"].isna()].copy()
missing_price[["title", "price_gbp", "rating", "availability", "source_page"]]

,title,price_gbp,rating,availability,source_page


## Exercise 2

Calculate average price by rating. Should products with missing price be included automatically?

In [62]:
# Your code here
avg_price_by_rating = (
    books
    .groupby("rating", dropna=False)["price_gbp"]
    .agg(["count", "mean"])
    .rename(columns={"count": "nonmissing_price_count", "mean": "average_price"})
)

avg_price_by_rating

,nonmissing_price_count,average_price
rating,,
1,9,34.385556
2,6,33.243333
3,9,39.120000
4,6,36.803333
5,10,31.652000


## Exercise 3

Use regex to extract the first number from `raw_availability`. Compare your result with the `available_count` column.

In [63]:
#Note that the listing page tells you whether the book is in stock; the detail page tells you how many are available.
availability_counts = []

for product_url in books["url"]:

    r = requests.get(product_url, timeout=5)
    r.raise_for_status()
    r.encoding = "utf-8"

    detail_soup = BeautifulSoup(r.text, "html.parser")
    availability_text = None

    for row in detail_soup.select("table.table.table-striped tr"):
        heading = row.select_one("th")
        value = row.select_one("td")

        if heading and heading.get_text(strip=True) == "Availability":
            availability_text = value.get_text(strip=True)
            break

    match = re.search(r"(\d+)\s+available", availability_text)

    availability_count = int(match.group(1)) if match else None
    availability_counts.append(availability_count)

    time.sleep(0.25)

books["availability_count"] = availability_counts

In [64]:
books.head()

,title,url,price_gbp,raw price,rating,availability,source_page,availability_count
0,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...,51.77,Â£51.77,3,In stock,1,22
1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...,53.74,Â£53.74,1,In stock,1,20
2,Soumission,https://books.toscrape.com/catalogue/soumissio...,50.10,Â£50.10,1,In stock,1,20
3,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...,47.82,Â£47.82,4,In stock,1,20
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...,54.23,Â£54.23,5,In stock,1,20


## Exercise 4

Create a final cleaned DataFrame that excludes unresolved duplicate URLs but retains rows with missing price as flagged records.

In [51]:
# Your code here
books["duplicate_url"] = books.duplicated(subset=["url"], keep=False)
duplicate_urls = books.loc[books["duplicate_url"], "url"].dropna().unique()

# Step 2: exclude all rows associated with unresolved duplicate URLs.
# We do NOT arbitrarily keep the first occurrence because the two listings may disagree.
final_books = books.loc[~books["url"].isin(duplicate_urls)].copy()

# Step 3: retain missing-price records but flag them.
final_books["missing_price_flag"] = final_books["price_gbp"].isna()

# Step 4: optional validation flag.
final_books["rating_issue_flag"] = (
    final_books["rating"].notna()
    & ~final_books["rating"].between(1, 5)
)

final_books[[
    "title", "url", "price_gbp", "rating", "availability",
    "missing_price_flag", "rating_issue_flag", "source_page"
]]

print("Original rows:", len(books))
print("Rows after excluding unresolved duplicate URLs:", len(final_books))
print("Missing-price rows retained:", int(final_books["missing_price_flag"].sum()))


Original rows: 60
Rows after excluding unresolved duplicate URLs: 60
Missing-price rows retained: 0


## Exercise 5 — Discussion

1. **What did the scraper assume about the HTML structure?**  
   It assumes each product is contained in `article.product-card`; title/URL are in `h3 a`; price is in `.price`; rating is in `.rating`; availability is in `.availability`; and pagination is represented by `a.next`.

2. **Which assumptions could fail if the website changes?**  
   Class names or tags could change, fields could move to different elements, prices could be loaded with JavaScript instead of appearing in the downloaded HTML, the rating format could change, a field could become optional, or the pagination link could use a different structure. The scraper may then return missing or incomplete data without necessarily producing an obvious Python error.

3. **Which validation checks would you include before sending this data to a manager?**  
   Check expected row counts, missingness in critical fields, duplicate product identifiers/URLs, price parsing success and plausible ranges, ratings between 1 and 5, URL completeness, availability parsing, and whether the number of product cards found per page is unexpectedly zero or sharply different from previous runs. Compare a small sample of scraped records with the actual webpage.

4. **What should the manager be told about how the data were collected?**  
   State the website/source, date and time of collection, pages/categories covered, fields extracted, any exclusions or unresolved duplicates, missing fields, known limitations, and that the data represent what was visible on the website at scraping time rather than necessarily the entire market or inventory.